# Comparing LLM Parameters with GPT-4o-mini

This notebook calls **OpenAI's `gpt-4o-mini`** model through the API and shows how different
generation parameters (`temperature`, `top_p`, `max_tokens`, `presence_penalty`, `frequency_penalty`)
change the output for the *same* prompt.

It reads your API key from **Colab Secrets** (the key icon 🔑 in the left sidebar), so your key is
never typed or hard-coded into the notebook.

### One-time setup (Colab Secrets)
1. Click the 🔑 **key icon** in the left sidebar of Colab.
2. Add a new secret named exactly `OPENAI_API_KEY`.
3. Paste your OpenAI API key as the value.
4. Toggle **"Notebook access"** ON for this notebook.


## 1. Install dependencies

In [ ]:
!pip install -q openai --upgrade


## 2. Load API key from Colab Secrets

In [ ]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("OPENAI_API_KEY")  # must match the secret name exactly

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

print("Client ready. Using model:", MODEL)


## 3. A helper to call the model with custom parameters

In [ ]:
def ask(prompt, temperature=1.0, top_p=1.0, max_tokens=150,
        presence_penalty=0.0, frequency_penalty=0.0):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        presence_penalty=presence_penalty,
        frequency_penalty=frequency_penalty,
    )
    return response.choices[0].message.content


## 4. Parameter #1 — `temperature`

Low temperature (close to 0) → focused, deterministic, repeatable output.
High temperature (close to 2) → more random, creative, sometimes less coherent output.


In [ ]:
prompt = "Write one creative opening sentence for a short story about a lighthouse."

for temp in [0.0, 0.7, 1.4]:
    print(f"=== temperature = {temp} ===")
    print(ask(prompt, temperature=temp))
    print()


## 5. Parameter #2 — `top_p` (nucleus sampling)

`top_p` limits sampling to the smallest set of tokens whose cumulative probability
reaches `top_p`. Lower `top_p` → narrower, safer word choices. Higher `top_p` → wider
variety of possible next words.


In [ ]:
prompt = "Describe a futuristic city in one paragraph."

for tp in [0.1, 0.5, 1.0]:
    print(f"=== top_p = {tp} ===")
    print(ask(prompt, temperature=1.0, top_p=tp))
    print()


## 6. Parameter #3 — `max_tokens`

Simply caps how long the response can be. Useful for controlling cost and length.


In [ ]:
prompt = "Explain how photosynthesis works."

for mt in [20, 80, 250]:
    print(f"=== max_tokens = {mt} ===")
    print(ask(prompt, temperature=0.7, max_tokens=mt))
    print()


## 7. Parameter #4 & #5 — `presence_penalty` and `frequency_penalty`

- `presence_penalty`: penalizes tokens that have already appeared at all → encourages
  talking about *new* topics.
- `frequency_penalty`: penalizes tokens based on *how often* they've appeared → discourages
  repeating the same words over and over.


In [ ]:
prompt = "List several benefits of regular exercise."

configs = [
    {"presence_penalty": 0.0, "frequency_penalty": 0.0},
    {"presence_penalty": 1.5, "frequency_penalty": 0.0},
    {"presence_penalty": 0.0, "frequency_penalty": 1.5},
]

for cfg in configs:
    print(f"=== presence_penalty={cfg['presence_penalty']}, frequency_penalty={cfg['frequency_penalty']} ===")
    print(ask(prompt, temperature=0.8, **cfg))
    print()


## 8. Side-by-side comparison table

Runs the same prompt across several parameter combinations and lays out the results in a table.

In [ ]:
import pandas as pd

prompt = "Give me a one-sentence tagline for a coffee shop."

param_sets = [
    {"label": "Low temp (0.0)",      "temperature": 0.0, "top_p": 1.0},
    {"label": "Medium temp (0.8)",   "temperature": 0.8, "top_p": 1.0},
    {"label": "High temp (1.5)",     "temperature": 1.5, "top_p": 1.0},
    {"label": "Low top_p (0.2)",     "temperature": 1.0, "top_p": 0.2},
]

rows = []
for p in param_sets:
    label = p.pop("label")
    output = ask(prompt, max_tokens=40, **p)
    rows.append({"setting": label, "output": output})

df = pd.DataFrame(rows)
df


## Notes

- If you see an authentication error, double-check that the secret is named exactly
  `OPENAI_API_KEY` and that **Notebook access** is toggled on for this notebook.
- `gpt-4o-mini` is used because it's fast and inexpensive — great for experimenting with
  parameters without racking up cost.
- Your API key never appears in the notebook text or output — it's only read into memory via
  `userdata.get(...)`.
